In [1]:
import sys
print(sys.version)
print(sys.executable)

3.9.6 (default, May 22 2026, 11:13:45) 
[Clang 21.0.0 (clang-2100.1.1.101)]
/Users/deslandes/Documents/MIT 6.00.1x/.venv/bin/python


In [ ]:
import torch
print(torch.__version__)

# hyperparameters

batch_size = 32
block_size = 8
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu' # whats is cuda?
eval_iters = 200
n_embed = 32


2.8.0


In [7]:
from urllib.request import urlretrieve

urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", "input.txt")

('input.txt', <http.client.HTTPMessage at 0x113368fd0>)

In [8]:
with open ('input.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()

In [ ]:
print (f"length of the string is {len(text):,}")


length of the string is 1,115,394


In [13]:
sorted_characters = sorted(list(set(text)))
vocab_size = len(sorted_characters)
print(''.join(sorted_characters))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [ ]:
#create a mapping from chracters to integers (tokenizer) using dictionaries
strtoint = { j:i for i,j in enumerate(sorted_characters)}
inttostr = { i:j for i,j in enumerate(sorted_characters)}

encoder = lambda s: [strtoint[vocab_size] for vocab_size in s] # lambda is a "quick" function that can be defined in one line. it takes a string input as s and then returns a list of integers that correspond to the characters in the string.
decoder = lambda l: ''.join(inttostr[d] for d in l) # changes a list of integers back to a string

print(encoder("hello world"))
print(decoder(encoder("hello world")))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [17]:
data = torch.tensor(encoder(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [18]:
#spltting into training and validation data sets

n = int(0.9*len(data))

training_data = data[:n]
validation_data = data[n:]

In [19]:
block_size = 8 # this is how many characters we are considering at once
x = training_data[:block_size]
y = training_data[1:block_size+1]




In [28]:
# this code will make training examples for predicting tokens
torch.manual_seed(1337) # random but repeatable 1337 sets the intial state
batch_size = 4 # one batch has four individual sequences of text that the model processes together 
block_size = 8 # each input sequence contains 8 characters / tokens

def get_batch(split): # defines a fucntion called get_batch that takes a string input called split, telling it with dataset to use 
    data = training_data if split == 'train' else validation_data # so calling get_batch('train') will use training dataset
    ix = torch.randint(len(data)-block_size, (batch_size,)) # this means pick four random integers from 0 through 91 each of which represents the startign of a sequence
    #(batch_size,) is a tuple that tells pytorch to return a tensor of the size batch_size 

    x = torch.stack([data[i:i+block_size] for i in ix]) # this means go to postion i in the data tensor and take the next block_size characters and do this for each of the four random integers in ix. 
    #torch.stack takes a list of tensors and combines them into a single tensor])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

input_tokens, target_tokens = get_batch('train') # pulls training data according tpo the above function

print('inputs:')
print(input_tokens.shape)
print(input_tokens)
print('targets:')
print(target_tokens.shape)
print(target_tokens)

print('----')

for batch_size in range(batch_size):
    for sequence_length  in range(block_size): # goes through each block in each batch
        context = input_tokens[batch_size, :sequence_length +1]
        target = target_tokens[batch_size, sequence_length ] # hence the prediction task is essentially "given [5,8,2,9,3,4] predict 7"
        print(f"when input is {context.tolist()} the target is {target}")       

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target is 43
when input is [24, 43] the target is 58
when input is [24, 43, 58] the target is 5
when input is [24, 43, 58, 5] the target is 57
when input is [24, 43, 58, 5, 57] the target is 1
when input is [24, 43, 58, 5, 57, 1] the target is 46
when input is [24, 43, 58, 5, 57, 1, 46] the target is 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target is 39
when input is [44] the target is 53
when input is [44, 53] the target is 56
when input is [44, 53, 56] the target is 1
when input is [44, 53, 56, 1] the target is 58
when input is [44, 53, 56, 1, 58] the target i

In [29]:
input_tokens, target_tokens = get_batch('train')
print(input_tokens.shape)

torch.Size([3, 8])


In [ ]:
# buidling a simple bigram model - predicts the next token based on the current token

import torch.nn as nn # pytorch's neural network library
from torch.nn import functional as F # makes the functions in torch.nn.functional available as F

class BigramLanguageModel(nn.Module): # inherits from nn.Module pytorch's model mahcinery like tracking trainable parameters
    def __init__(self, n_embed): # vocab_size tells the function how many distinct charcaters exist
        super().__init__() # initialize the nn.Module

        self.token_embedding_table = nn.Embedding (vocab_size, n_embed)
        # the above creates a square table (vocab_size x vocab_size) which is a trainable lookup tbale 
        # this is a type of neural network layer
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets = None): # idx takes input_tokens and input tokens and targets takes target_tokens as the correct answers 
        token_embeddings = self.token_embedding_table(idx) # looks up a row of scores for every input token in idx
        logits = self.lm_head(token_embeddings) # why did we create a linear layer here?
        


        # these are in the shape (batch_size, sequence_length , vocab_size)
        # batch_size: number of sequences 
        # sequence_length : tokens (positions) per sequences
        # vocab_size: vocab size

        if targets is None:
            loss = None # so if we don have correct answers we dont calculate the loss
        else:
            batch_size,sequence_length ,vocab_size = logits.shape #pulls out the shape of the logits tensor 
            logits = logits.view(batch_size*sequence_length , vocab_size) # rehapes ths logits tensor into a 2D table with batch_size*sequence_length  rows and vocab_size columns
            # this puts the predictions in the format of one row of scores per prediction

            targets = targets.view(batch_size*sequence_length ) # reshapes the targets tensor into a 1D vector with batch_size*sequence_length  elements
            loss = F.cross_entropy(logits, targets) # measure how much probabality the moeel assigns to the correect next towkn at eahc postion 
        return logits, loss

    def generate(self, token_sequence, max_new_tokens): # this is what will extend the startign sequence by generating tokens
        # self is the model isntance
        # token_sequence is the starting context of indices (tokens)
        # max_new_tokens is how many new tokens to generate
        for _ in range (max_new_tokens): # means we are repeating the work a max_new_tokens number of times.
            # the _ means we are ignoring the loop variable since we dont need it
            logits, loss = self(token_sequence) # self is the model instance, this runs the forwrad pass (wihout targets we a re only generating tet withoutchekcing against known ansers)
            logits = logits[:, -1, :] # takes the last oof the logits sequence and the : means we are taking everything along this dimension
            probabilities = F.softmax(logits, dim = -1) # softmax turns the sores for sequence_length  into probabilities for each possible next character
            # dim = -1 tells softmax to use the last dimension of logits which conatins the scores for each possible next character
            next_token = torch.multinomial(probabilities, num_samples = 1) # multinomial makes a random draw when each sample has dfferent proablities of being picked and num_samples tells it how many to draw
            token_sequence = torch.cat((token_sequence, next_token), dim = 1) # concatenates the new oekn aong the dimension of the sequence so that the new token is added to the end of the sequence
        return token_sequence 

m = BigramLanguageModel(vocab_size) # creates a model instance
logits, loss = m(input_tokens, target_tokens) # passing in the sequences with the correct answers to run the forward pass
# this only runs the forward pass because pytorch defines calling a model witt parentheses as calling the forward method
print(logits.shape) # this prints the reurned scores dimensions (prediction posions each with 65 charcater scores)
print(loss) # this tells you how correct the model was across the prediction postions.

print(decoder(m.generate(token_sequence = torch.zeros((1,1), dtype = torch.long), max_new_tokens = 100) [0].tolist()))
# this starts witll one token and then geenrates 100 from it and changes it to a python list from a tensor with zero sleecting the first sequence which is passed into decoder


torch.Size([24, 65])
tensor(4.7448, grad_fn=<NllLossBackward0>)

.PUOLpMZ&&Ti3iPObym WpYwz3eVev&uZm:OOdHPDoMwZ?qIFXCo-TScc3!t3yvLfUXDoTX
ZT!Uh'accMX&HaBQH&cNKEn&
Ge'


In [35]:
# now create a pytorch optimizer which will update the graidents of the model parameters

optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3) # different from standard adam in how it handles weight decay

In [37]:
batch_size = 32 # this tells how many input sequences we train together in one batch 
for steps in range(100): # run the loop 100 times
    
    # sample a batch of data
    input_tokens, target_tokens = get_batch('train')

    #evaluate the loss
    logits, loss = m(input_tokens, target_tokens) # runs forward pass method
    optimizer.zero_grad(set_to_none = True) # clears leftover gradients 
    loss.backward()
    optimizer.step()

print(loss.item())

4.505012512207031


In [38]:
print(decoder(m.generate(token_sequence = torch.zeros((1,1), dtype = torch.long), max_new_tokens = 500) [0].tolist()))


AuUYayd:
zOP

N?sCTd:xWynrKlhTFJJjH?O:
EuS;b'o-ui3eIKES;
C,IPH&:LzqUUGK!id:uSVam$;Xwi3nfW!-Hrx&pQEbwcVgh:nrN.WFVCFAW?-v&3.r-OCg$CoyFeBVu&PuSR
BJGgc3O:OdYDE3wak;tae,Q&$?YQmg3ffmK&CDUGEuC:NPO!Kyk;aCaM,aCaOldikb NtO'h'Z-&v&cc3w, ZKcc3P;OK!Oq,WJt;.t:&ReF-Pk;f3Iede;
lNErEb,pz3wZClNDQG-REUAoW!HP ROB,azZOi.Sg3P -mj.SwP:oo-:SeF3 ymKA$vpI!FR
WABVQD-RBsFEGVM fC'vo$3mKy!kQZjj?Gryv&ELAhNPi$FxNj-,Z?md.NXPj;ff.Eyvx.hYcuUOXua'hP,KW?WifvIeMUUOwzDH:3;X.ByrwqO.OdIec3.SoH!'h Kiv&T,bnufAPg
L'aCgi3m,du
pMXlCpoJ
VC-:


In [53]:
# getting into the attention mechanism - example below

torch.manual_seed(42)

a = torch.tril(torch.ones(3,3)) # torch.ones creates a matrix of ones and .tril makes it lower triangular 
# matrix b stand for the token infromation while a conatins the weights telling us how to combine it 
a = a / torch.sum(a, 1, keepdim = True) # divding each row by its sum, turning the ones into averaging weights

b = torch.randint(0,10,(3,2)).float() # creates 3 columns and 2 rows of random integers 0 through 9

c = a @ b # uses a's weights to combine b's infromation 

print(f"a = {a}")
print(f"b = {b}")
print(f"c = {c}")

# example of how we'd create a matrix using our modelo data

torch.manual_seed(1337)
xbatch_size, xsequence_length, xfeature_dim = 4, 8, 2
example_tensor = torch.randn(xbatch_size, xsequence_length, xfeature_dim)
example_tensor.shape

# version 1
# essentially this code walks through each sequence and says take the average of the feature dimension with the numbers seen so far in a new vector
# the purpose of this is each poiotion has a summary of what came before it 
prefixed_averages = torch.zeros((xbatch_size, xsequence_length, xfeature_dim))
for size in range(xbatch_size):
    for length in range(xsequence_length):# visiting eahc token in each sequence 
        sequence = example_tensor[size,:length+1]
        prefixed_averages[size, length] = torch.mean(sequence, 0)

## ______

# version 2
# a different way to compute the same avergaing operation 

wei = torch.tril(torch.ones(xsequence_length, xsequence_length)) # creates one row and one column per token position
wei = wei / wei.sum(1, keepdim = True)
# each row represents a positon we're computing a summary for
# each columun represnets an input positin that could contribute
prefixed_averages2 = wei @ example_tensor # this multiplies the avergaing matrix be each sequence's feature table 

print(torch.allclose(prefixed_averages, prefixed_averages2)) #now comparing the two versions 

## ______

# version 3

tril = torch.tril(torch.ones(xsequence_length, xsequence_length))
wei = torch.zeros((xsequence_length, xsequence_length))
wei = wei.masked_fill(tril == 0, float('-inf')) # masked_fill(condition, value) replaces entries wherever the conditoon is true. this makes the future postions negative infinity so softmax makes zero
wei = F.softmax(wei, dim=-1) # computes softmax, ie, exponentiates and divides by the row total
prefixed_averages3 = wei @ example_tensor

torch.allclose(prefixed_averages, prefixed_averages3)

## ______

# version 4 - employing self attention 

torch.manual_seed(1337)
xbatch_size, xsequence_length, xfeature_dim = 4,8,32
example_tensor = torch.randn(xbatch_size, xsequence_length, xfeature_dim) # each sequence is a 8 x 32 table

head_size = 16 # chooses 16 numbers per token for tge key, query and value vectors that are used in attention

# given each token has 32 parameters currently we'll use separate transormations to turn this into (each with length 16):

# a query: used to compare a position with other positions
# a key: what other postions' queries comapre against 
# a value: the combination of the resulting attention weights

key = nn.Linear(xfeature_dim, head_size, bias = False)
query = nn.Linear(xfeature_dim, head_size, bias = False)
value = nn.Linear(xfeature_dim, head_size, bias = False) # each of these transformations happen with pytorch's own radnomly initialized weights 

k = key(example_tensor) 
q = query(example_tensor)

wei = q @ k.transpose(-2, -1) #multplying the two vectors to create the intial matrix to do softmax over

tril = torch.tril(torch.ones(xsequence_length, xsequence_length))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = -1)

v = value(example_tensor)
weighted_output_tensor = wei @ v # takes the values and applies the "attention" to them 

weighted_output_tensor.shape





a = tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b = tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c = tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
True


torch.Size([4, 8, 16])